> Notebook-friendly copy of `part-I/1.3-numpy-solutions.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

In [ ]:
# --- environment setup (generated, not part of the lesson) ---
# Colab and Kaggle do not ship every package this notebook imports.
# Colab and Kaggle start in an empty working directory.
# This is a no-op in an environment that is already set up.
import importlib.util
import subprocess
import sys

for module, package in {"gsw": "gsw", "pooch": "pooch"}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

from pathlib import Path

Path("_files").mkdir(exist_ok=True)   # the folder this notebook writes into

# Solutions

**ℹ️ Reference solutions**

Worked solutions for every exercise in [1.3-numpy-exercises.ipynb](1.3-numpy-exercises.ipynb), including the real Argo walkthrough. Exercise 9 needs the `gsw` library, which is part of the book's environment.

## Exercise 1: Create and inspect

Build the following 3×4 field of temperatures (°C) as a numpy array:

`5.2 4.8 6.1 3.9 1.3 0.5 -0.8 -1.2 -2.6 -3.1 -1.9 0.2`

Print its `shape`, `ndim`, and `dtype`, and its overall mean rounded to two decimals.
Then create, and print, three arrays of the same shape without typing the shape out by hand:
all ones, a uniform reference field of 15.0 °C, and random values in [0, 1).

In [ ]:
import numpy as np

field_celsius = np.array([
    [ 5.2,  4.8,  6.1,  3.9],
    [ 1.3,  0.5, -0.8, -1.2],
    [-2.6, -3.1, -1.9,  0.2],
])
print(field_celsius.shape, field_celsius.ndim, field_celsius.dtype)
print(round(float(field_celsius.mean()), 2))

print(np.ones(field_celsius.shape))
print(np.full(field_celsius.shape, 15.0))       # reference field, °C
print(np.random.random(field_celsius.shape))

## Exercise 2: Index and slice

Given

```python
a = np.array([[1.0, 2.0, 3.0, 4.0],
              [5.0, 6.0, 7.0, 8.0],
              [9.0, 10.0, 11.0, 12.0]])
```

print the last row, the first column, and the 2×2 sub-block formed by the first two rows and the last two columns. 

Finally, take a slice of the first row, change its first element to `99.0`, and print the original array. Then repeat using `.copy()`. Explain the difference in a comment.

In [ ]:
import numpy as np

In [ ]:
a = np.array([[1.0, 2.0, 3.0, 4.0],
              [5.0, 6.0, 7.0, 8.0],
              [9.0, 10.0, 11.0, 12.0]])
print(a[-1, :])
print(a[:, 0])
print(a[0:2, 2:4])

In [ ]:
row = a[0, :]       # a slice is a view, not a copy
row[0] = 99.0
print(a)            # the original changed too

In [ ]:
b = np.array([[1.0, 2.0, 3.0, 4.0],
              [5.0, 6.0, 7.0, 8.0],
              [9.0, 10.0, 11.0, 12.0]])
row_copy = b[0, :].copy()
row_copy[0] = 99.0
print(b)            # unaffected: .copy() breaks the shared memory

## Exercise 3: Masking and np.where

Given

```python
temp_celsius = np.array([[-1.0, 12.0,  3.0],
                         [ 8.0, -2.0, 15.0],
                         [ 4.0,  0.5, 11.0]])
```

print the values that exceed 10 °C, their mean, and the percentage of the field they make up.
Then build a clipped field in which every value below 0 °C is set to 0, using `np.where`.

In [ ]:
import numpy as np

temp_celsius = np.array([[-1.0, 12.0,  3.0],
                         [ 8.0, -2.0, 15.0],
                         [ 4.0,  0.5, 11.0]])
mask = temp_celsius > 10.0
print(temp_celsius[mask])
print(temp_celsius[mask].mean())
print(f"{100.0 * mask.sum() / temp_celsius.size:.1f} % of the field")

clipped = np.where(temp_celsius < 0.0, 0.0, temp_celsius)
print(clipped)

## Exercise 4: Broadcasting and grids

Given the field below, add a per-column offset (length 4) to every row, and separately add a per-row offset (length 3) to every column.

```python
field = np.array([[1.0, 2.0, 3.0, 4.0],
                  [5.0, 6.0, 7.0, 8.0],
                  [9.0, 10.0, 11.0, 12.0]])
col_offset = np.array([0.0, 1.0, 2.0, 3.0])
row_offset = np.array([10.0, 20.0, 30.0])
```

Then build a grid of 4 evenly spaced longitudes from 6 to 9 °E and 3 evenly spaced latitudes
from 46 to 47 °N, so that it has the same shape as `field`. On that grid, compute a synthetic temperature
field that is 15.0 °C at (6 °E, 46 °N), cools by 0.6 °C per degree of latitude northward, and warms
by 0.2 °C per degree of longitude eastward. Compute it twice, once from the 2D arrays that
`np.meshgrid` returns and once by broadcasting the 1D latitudes and longitudes directly, and
confirm with `np.allclose` that the two agree.

In [ ]:
import numpy as np

In [ ]:
field = np.array([[1.0, 2.0, 3.0, 4.0],
                  [5.0, 6.0, 7.0, 8.0],
                  [9.0, 10.0, 11.0, 12.0]])
col_offset = np.array([0.0, 1.0, 2.0, 3.0])
row_offset = np.array([10.0, 20.0, 30.0])

In [ ]:
print(field + col_offset)            # (3,4) + (4,) broadcasts across rows
print(field + row_offset[:, None])   # (3,4) + (3,1) broadcasts across columns

In [ ]:
lon = np.linspace(6.0, 9.0, 4)     # 4 longitudes, °E
lat = np.linspace(46.0, 47.0, 3)   # 3 latitudes, °N

In [ ]:
lon2d, lat2d = np.meshgrid(lon, lat)                              # both (3, 4)
temp_grid_celsius = 15.0 - 0.6 * (lat2d - 46.0) + 0.2 * (lon2d - 6.0)

In [ ]:
# the same field without meshgrid: a (3, 1) column plus a (4,) row broadcasts to (3, 4)
temp_bcast_celsius = 15.0 - 0.6 * (lat[:, None] - 46.0) + 0.2 * (lon - 6.0)

In [ ]:
print(temp_grid_celsius)
print(temp_grid_celsius.shape, temp_bcast_celsius.shape)         # (3, 4) (3, 4)
print(np.allclose(temp_grid_celsius, temp_bcast_celsius))        # True

## Exercise 5: Axis reductions

Given seasonal mean temperatures at three stations, one row per station and one column per season
(winter, spring, summer, autumn):

```python
seasonal_celsius = np.array([[ 2.1,  9.8, 18.4, 10.2],
                             [-1.5,  7.2, 16.9,  8.0],
                             [ 4.0, 11.5, 20.3, 12.6]])
```

print the annual mean at each station and the index of each station's warmest season. Then find the
coldest value in each season twice, once with the method and once with the numpy function, and
confirm they agree.

Finally, given four days of rainfall at two stations, one row per station,

```python
rain_mm = np.array([[0.0, 6.2, 1.1, 0.0],
                    [2.4, 0.0, 0.0, 8.3]])
```

use `cumsum` along the right axis to get the rain to date. Print the index of the station that was
wetter after the second day and the index of the station that was wetter after all four. Are they
the same station?

In [ ]:
import numpy as np

In [ ]:
seasonal_celsius = np.array([[ 2.1,  9.8, 18.4, 10.2],
                             [-1.5,  7.2, 16.9,  8.0],
                             [ 4.0, 11.5, 20.3, 12.6]])

In [ ]:
print(seasonal_celsius.mean(axis=1))      # annual mean at each station
print(seasonal_celsius.argmax(axis=1))    # [2 2 2]: summer at every station

In [ ]:
print(seasonal_celsius.min(axis=0))       # coldest value in each season, as a method
print(np.min(seasonal_celsius, axis=0))   # the same, as a numpy function

In [ ]:
rain_mm = np.array([[0.0, 6.2, 1.1, 0.0],
                    [2.4, 0.0, 0.0, 8.3]])
rain_to_date_mm = rain_mm.cumsum(axis=1)  # stations are rows, so accumulate along axis 1
print(rain_to_date_mm)
print(int(rain_to_date_mm[:, 1].argmax()))    # 0: wetter after day 2
print(int(rain_to_date_mm[:, -1].argmax()))   # 1: wetter after day 4, a different station

## Exercise 6: Missing data and interpolation

Given hourly air temperatures with a gap of two readings,

```python
temp_celsius = np.array([12.0, 13.5, np.nan, np.nan, 18.0, 18.5])
```

fill the gap by linear interpolation against the integer index. Then print the NaN-aware mean of the
original series and the ordinary mean of the filled one. The two differ; say in a comment why.

In [ ]:
import numpy as np

temp_celsius = np.array([12.0, 13.5, np.nan, np.nan, 18.0, 18.5])
hours = np.arange(temp_celsius.size)
present = ~np.isnan(temp_celsius)             # True where a reading exists

filled_celsius = np.interp(hours, hours[present], temp_celsius[present])
print(filled_celsius)                         # [12.  13.5 15.  16.5 18.  18.5]

print(np.nanmean(temp_celsius))               # 15.5: the four real readings
print(filled_celsius.mean())                  # 15.58...: six values, two of them filled
# np.nanmean averages only the four real readings. Filling adds two values on the straight
# line from 13.5 to 18.0; they average 15.75, above the 15.5 of the real readings, so they
# pull the mean up. np.nanmean ignores the gap, while the filled mean assumes the temperature
# changed linearly across it.

## Exercise 7: Avoid the dtype trap

An assistant wrote this function to compute each cell's share of the total rainfall in an integer
field of rain-gauge readings:

```python
def rain_share(rain_mm):
    share = np.zeros_like(rain_mm)
    share[:] = rain_mm / rain_mm.sum()
    return share

rain_mm = np.array([[0, 2, 5], [1, 0, 8], [3, 4, 2]])
```

Call it and print the result. Explain in a comment why every share is 0, then fix the function
twice: once keeping the preallocation but passing `dtype=float` to `np.zeros_like`, and once
without preallocating at all. Confirm that each fixed version returns a float array whose shares
sum to 1.

In [ ]:
import numpy as np

In [ ]:
def rain_share(rain_mm):
    share = np.zeros_like(rain_mm)
    share[:] = rain_mm / rain_mm.sum()
    return share

In [ ]:
rain_mm = np.array([[0, 2, 5], [1, 0, 8], [3, 4, 2]])
print(rain_share(rain_mm))
# np.zeros_like copies the integer dtype of rain_mm. Every share lies between 0 and 1, so
# assigning it into the integer array truncates it to 0.

In [ ]:
def rain_share_float(rain_mm):
    share = np.zeros_like(rain_mm, dtype=float)   # preallocate as float explicitly
    share[:] = rain_mm / rain_mm.sum()
    return share

In [ ]:
def rain_share_direct(rain_mm):
    return rain_mm / rain_mm.sum()                # division already returns floats

In [ ]:
share_float = rain_share_float(rain_mm)
share_direct = rain_share_direct(rain_mm)
# the sums come out as 1.0000000000000002, so compare with np.allclose, not ==
print(share_float.dtype, np.allclose(share_float.sum(), 1.0))     # float64 True
print(share_direct.dtype, np.allclose(share_direct.sum(), 1.0))   # float64 True

## Exercise 8: Reshape and stack

1. Create the integers 0 to 11 as a 1D array, then reshape it into a 3×4 array and print both
   shapes.
2. Reshape the same data into a 4×3 array. Print it and say in a comment why the numbers appear
   in a different arrangement.
3. Use `reshape(-1)` to flatten your 3×4 array back to 1D, and confirm the shape.
4. Stack the 3×4 array on top of a row of zeros with `np.vstack`, and print the resulting shape.

In [ ]:
import numpy as np

flat = np.arange(12)
grid = flat.reshape(3, 4)
print(flat.shape, grid.shape)          # (12,) (3, 4)

other = flat.reshape(4, 3)
print(other)
# the data is stored in one flat block and read row by row, so changing the shape
# changes where each row ends, not the order of the values

print(grid.reshape(-1).shape)          # (12,) — -1 infers the length

stacked = np.vstack([grid, np.zeros(4)])
print(stacked.shape)                   # (4, 4)

## Exercise 9: Ocean float profiles

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/argo-float-cycle.png" alt="A diagram of one Argo float cycle, numbered one to seven, from deployment through drift, dive, ascent and satellite transmission" width="100%">

<em>One cycle of an Argo float: deployment, descent to a drifting depth of about 1000 m, ten days of
drift, a dive to the profiling depth, then the ascent during which the measurements are taken, and
transmission to satellite at the surface before the next cycle begins. The panel on the right shows
what one ascent produces — the temperature and salinity profiles you are about to work with.
Illustration &copy; Thomas Haessig, from
<a href="https://www.euro-argo.eu/Outreach/Educational-material/Discover-Argo-floats-for-kids/How-do-the-Argo-floats-fulfill-their-mission-in-the-Ocean">Euro-Argo ERIC's educational material</a>.</em>


[Argo](https://argo.ucsd.edu/) floats are autonomous instruments that drift with ocean currents,
periodically diving and surfacing while recording temperature, salinity, and pressure. Each dive
produces one *profile*: a set of measurements at many depth *levels*, with a single latitude,
longitude, and date attached.

This exercise uses real Argo data. Steps 1 to 8 use only what 1.1 to 1.3 cover: loading
arrays, inspecting shapes, rebuilding an axis, vectorised arithmetic, reductions along an axis,
boolean masks, and NaN-aware statistics. Steps 9 to 11 then plot what you computed.

**ℹ️ Beyond this subchapter**

Steps 9 to 11 use `matplotlib`, which gets its own treatment in 1.4. Import it once with
`import matplotlib.pyplot as plt`; after that you need only the functions below, and each is
named again where it is wanted:

- `plt.figure()` — start a new, empty figure; without it, every plot in a cell lands on the
  same axes
- [`plt.plot`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html) — a line, or
  one line per column when handed a 2D array
- [`plt.errorbar`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.errorbar.html) — a
  line with error bars
- [`plt.scatter`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.scatter.html) — points
- `plt.xlabel`, `plt.ylabel`, `plt.title` — each takes a string
- `plt.show()` — draw the current figure

An array of numbers with no picture attached is hard to sanity-check, which is why these three
steps sit here rather than waiting for 1.4: the profiles are the fastest way to see whether Steps 4
to 6 produced something physical.

In [ ]:
# Pre-supplied: download and unzip the Argo data files.
import pooch

files = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/argo_float_data.zip",
    known_hash="sha256:dd913ba3450241bcd5d5697187ea68d398bdd7fd563a04cc74386834749c25f5",
    processor=pooch.Unzip(),
    path=pooch.os_cache("mlees"),
)
for f in sorted(files):
    print(f)

**Step 1.** Load each array, taking the name from the filename.

In [ ]:
import numpy as np
from pathlib import Path

data_dir = Path(files[0]).parent           # every file was unzipped into this one folder

T = np.load(data_dir / "T.npy")            # temperature, °C
S = np.load(data_dir / "S.npy")            # practical salinity, unitless
P = np.load(data_dir / "P.npy")            # pressure, dbar
date = np.load(data_dir / "date.npy")      # one date per profile
lat = np.load(data_dir / "lat.npy")
lon = np.load(data_dir / "lon.npy")
level = np.load(data_dir / "levels.npy")   # depth level index

**Step 2.** The shapes tell you how the arrays fit together.

In [ ]:
for name, arr in (("T", T), ("S", S), ("P", P),
                  ("date", date), ("lat", lat), ("lon", lon), ("level", level)):
    print(name, arr.shape)

# T, S and P are (78, 75). Their first axis, of length 78, is the one they share with
# level; their second axis, of length 75, is the one they share with date, lat and lon.
# So axis 0 runs down through depth and axis 1 runs across profiles.

**Step 3.** `level` is a regular sequence, so it can be rebuilt two ways.

In [ ]:
print(level[0], level[-1], level.size)

level_arange = np.arange(level[0], level[-1] + 1)          # stop is excluded
level_linspace = np.linspace(level[0], level[-1], level.size)   # both ends included

print(np.allclose(level_arange, level))
print(np.allclose(level_linspace, level))

**Step 4.** The density anomaly, as one vectorised expression.

In [ ]:
# Pre-supplied: the conservative-temperature function from the gsw library
from gsw import CT_from_t

In [ ]:
a = 7.718e-1
b = -8.44e-2
c = -4.561e-3

conservative_temp = CT_from_t(S, T, P)          # salinity, temperature, pressure
# S is practical salinity standing in for absolute salinity: about 0.13 kg m-3 too light
relative_density = a * S + b * conservative_temp + c * conservative_temp**2

print(relative_density.shape, T.shape)
print(relative_density.shape == T.shape)

**Step 5.** Reduce across profiles, which is axis 1, leaving one value per depth.

In [ ]:
mean_T = T.mean(axis=1)
mean_S = S.mean(axis=1)
mean_P = P.mean(axis=1)
mean_density = relative_density.mean(axis=1)

std_T = T.std(axis=1)
std_S = S.std(axis=1)
std_P = P.std(axis=1)
std_density = relative_density.std(axis=1)

print(mean_T.shape, level.shape)
print("first three mean temperatures:", mean_T[:3])

**Step 6.** Those means are full of `nan`, because an ordinary mean propagates missing values.

In [ ]:
n_missing = np.isnan(T).sum()
print("missing values in T:", n_missing, "of", T.size)
print(f"that is {100.0 * n_missing / T.size:.1f} % of the array")

mean_T = np.nanmean(T, axis=1)
mean_S = np.nanmean(S, axis=1)
mean_P = np.nanmean(P, axis=1)
mean_density = np.nanmean(relative_density, axis=1)

std_T = np.nanstd(T, axis=1)
std_S = np.nanstd(S, axis=1)
std_P = np.nanstd(P, axis=1)
std_density = np.nanstd(relative_density, axis=1)

print("nan left in the mean temperature:", np.isnan(mean_T).sum())   # 0

**Step 7.** Label each depth, then select depths by mask.

In [ ]:
labels = np.where(mean_T > 10.0, "warm", "cold")
print(labels[:10])
print(level[:10])

cold_mask = mean_T < 5.0
print("depth levels colder than 5 °C:", level[cold_mask])

**Step 8.** Stack the summary, save it, and check the round trip.

In [ ]:
summary = np.vstack([level, mean_T, mean_S, mean_density])
print(summary.shape)

Path("_files").mkdir(exist_ok=True)
np.save("_files/argo_summary.npy", summary)
reloaded = np.load("_files/argo_summary.npy")
print(np.allclose(summary, reloaded))

**Step 9.** One line per profile, variable against depth.

In [ ]:
import matplotlib.pyplot as plt

for values, label in [
    (T, "Temperature (in degC)"),
    (S, "Salinity (in g/kg)"),
    (P, "Pressure (in dbar)"),
    (relative_density, "Relative density (in kg/m3)"),
]:
    plt.figure()
    plt.plot(values, level)      # values is (level, profile): one line per column
    plt.xlabel(label)
    plt.ylabel("Depth level")
    plt.title(f"{label.split(' (')[0]} profiles at different locations")
    plt.show()

**Step 10.** The mean profile of each variable, with the spread as horizontal error bars.

In [ ]:
for mean_values, std_values, label in [
    (mean_T, std_T, "Temperature (in degC)"),
    (mean_S, std_S, "Salinity (in g/kg)"),
    (mean_P, std_P, "Pressure (in dbar)"),
    (mean_density, std_density, "Relative density (in kg/m3)"),
]:
    plt.figure()
    # the variable is on the x-axis, so the spread is a horizontal bar: xerr, not yerr
    plt.errorbar(mean_values, level, xerr=std_values)
    plt.xlabel(f"{label} including simple error bars")
    plt.ylabel("Depth level")
    plt.title(f"Mean {label.split(' (')[0].lower()} profile")
    plt.show()

# For temperature, salinity and density the bars are widest near the surface and narrow
# with depth. These 75 profiles span two years, and the upper ocean is where the seasons and
# the float's changing position make the most difference; the deep water is nearly the same
# in every profile. Pressure is the exception: its spread stays below 1.6 dbar at every level
# and does not shrink with depth, because each level is recorded at almost the same pressure
# in every profile.

**Step 11.** Where and when the profiles were taken.

In [ ]:
plt.figure()
plt.scatter(lon, lat, s=20)
plt.xlabel("Longitude (in degrees east)")
plt.ylabel("Latitude (in degrees north)")
plt.title("Where each profile was taken")
plt.show()

print(date[:3])
print(date[-1])

# The dates are ten days apart and run from July 2012 to July 2014: 75 profiles on a steady
# ten-day cycle come from one float, not several. The points are that float's positions over
# the two years, spread over about 7 degrees of latitude (40-47 °N) and 8 of longitude
# (31-39 °W) in the North Atlantic. Because every profile comes from this one region, the deep
# parts of the mean profiles above agree so closely.